In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(data_dir, transform=transform)

print("Classes:", dataset.classes)

labels = [dataset.samples[i][1] for i in range(len(dataset))]
print("Class distribution:", np.bincount(labels))


In [ ]:
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=np.array(labels)[temp_idx],
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


In [ ]:
model = models.efficientnet_b0(weights="IMAGENET1K_V1")

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 1)
)

model = model.to(device)


In [ ]:
num_normal = np.sum(np.array(labels) == 0)
num_stroke = np.sum(np.array(labels) == 1)

print("Normal:", num_normal)
print("Stroke:", num_stroke)

pos_weight_value = num_normal / num_stroke
pos_weight = torch.tensor([pos_weight_value]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=5e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)


In [ ]:
history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "val_auc": [], "val_f1": []
}

epochs = 30

for epoch in range(epochs):

    # -------- TRAIN --------
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        loss = criterion(outputs, labels_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        train_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        train_preds.extend(preds.cpu().numpy().ravel())
        train_labels.extend(labels_batch.cpu().numpy().ravel())

    train_acc = accuracy_score(train_labels, train_preds)

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0
    val_preds, val_labels, val_probs = [], [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            val_preds.extend(preds.cpu().numpy().ravel())
            val_labels.extend(labels_batch.cpu().numpy().ravel())
            val_probs.extend(probs.cpu().numpy().ravel())

    val_preds = np.array(val_preds)
    val_labels = np.array(val_labels)
    val_probs = np.array(val_probs)

    val_acc = accuracy_score(val_labels, val_preds)

    if len(np.unique(val_labels)) > 1:
        val_auc = roc_auc_score(val_labels, val_probs)
    else:
        val_auc = 0.0

    val_f1 = f1_score(val_labels, val_preds)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss/len(train_loader))
    history["val_loss"].append(val_loss/len(val_loader))
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | "
          f"Val AUC: {val_auc:.4f}")


In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels_batch in test_loader:
        images = images.to(device)
        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy().ravel())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs.cpu().numpy().ravel())

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(all_labels, all_preds, target_names=["Normal","Stroke"]))

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("AUC:", roc_auc_score(all_labels, all_probs))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Normal","Stroke"],
            yticklabels=["Normal","Stroke"])
plt.title("Confusion Matrix")
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve")
plt.show()


EfficientNet + CBAM Model


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(data_dir, transform=transform)

print("Classes:", dataset.classes)

labels = [dataset.samples[i][1] for i in range(len(dataset))]
print("Class distribution:", np.bincount(labels))

In [ ]:
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=np.array(labels)[temp_idx],
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * out


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(x_cat))
        return x * out


class CBAM(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.ca = ChannelAttention(in_channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

In [ ]:
class EfficientNet_CBAM(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.efficientnet_b0(weights="IMAGENET1K_V1")

        self.features = backbone.features
        in_channels = 1280  # final feature channels

        self.cbam = CBAM(in_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(in_channels, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = EfficientNet_CBAM().to(device)

In [ ]:
num_normal = np.sum(np.array(labels) == 0)
num_stroke = np.sum(np.array(labels) == 1)

pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=5e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

In [ ]:
history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "val_auc": [], "val_f1": []
}

epochs = 30

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)

        # Stability clamp
        outputs = torch.clamp(outputs, -50, 50)

        loss = criterion(outputs, labels_batch)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        train_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        train_preds.extend(preds.cpu().numpy().ravel())
        train_labels.extend(labels_batch.cpu().numpy().ravel())

    train_acc = accuracy_score(train_labels, train_preds)

    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_loss = 0
    val_preds, val_labels, val_probs = [], [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            val_preds.extend(preds.cpu().numpy().ravel())
            val_labels.extend(labels_batch.cpu().numpy().ravel())
            val_probs.extend(probs.cpu().numpy().ravel())

    val_preds = np.array(val_preds)
    val_labels = np.array(val_labels)
    val_probs = np.array(val_probs)

    val_acc = accuracy_score(val_labels, val_preds)

    if len(np.unique(val_labels)) > 1:
        val_auc = roc_auc_score(val_labels, val_probs)
    else:
        val_auc = 0.0

    val_f1 = f1_score(val_labels, val_preds)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss/len(train_loader))
    history["val_loss"].append(val_loss/len(val_loader))
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | "
          f"Val AUC: {val_auc:.4f}")

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels_batch in test_loader:
        images = images.to(device)

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy().ravel())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs.cpu().numpy().ravel())

print("\n📊 CLASSIFICATION REPORT:\n")
print(classification_report(all_labels, all_preds, target_names=["Normal","Stroke"]))

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1 Score:", f1_score(all_labels, all_preds))
print("AUC:", roc_auc_score(all_labels, all_probs))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Normal","Stroke"],
            yticklabels=["Normal","Stroke"],
            cmap="Blues")

plt.title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

In [ ]:
precision_vals, recall_vals, _ = precision_recall_curve(all_labels, all_probs)

plt.plot(recall_vals, precision_vals)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="Train")
plt.plot(history["val_acc"], label="Val")
plt.title("Accuracy")
plt.legend()

plt.show()

Grad-CAM Implementation


In [ ]:
!pip install opencv-python

In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        
        self.gradients = None
        self.activations = None
        
        self.hook_layers()

    def hook_layers(self):
        def forward_hook(module, input, output):
            self.activations = output
        
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]
        
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate(self, input_image, class_idx=None):
        self.model.eval()
        
        output = self.model(input_image)
        
        if class_idx is None:
            class_idx = torch.argmax(output)
        
        self.model.zero_grad()
        output[:, class_idx].backward()

        gradients = self.gradients[0]
        activations = self.activations[0]

        pooled_gradients = torch.mean(gradients, dim=[1,2])

        for i in range(activations.shape[0]):
            activations[i, :, :] *= pooled_gradients[i]

        heatmap = torch.mean(activations, dim=0).cpu().detach().numpy()
        heatmap = np.maximum(heatmap, 0)
        heatmap /= np.max(heatmap)

        return heatmap

In [ ]:
target_layer = model.features[-4]   
gradcam = GradCAM(model, target_layer)

In [ ]:
model.eval()

num_images = 2
count = 0

plt.figure(figsize=(12, 8))

for images, labels_batch in test_loader:
    for i in range(len(labels_batch)):

        if labels_batch[i] == 1:  # Stroke class

            input_image = images[i].unsqueeze(0).to(device)
            original_image = images[i]

            heatmap = gradcam.generate(input_image)

            # Convert tensor image to numpy
            img = original_image.permute(1,2,0).cpu().numpy()
            img = img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
            img = np.clip(img, 0, 1)

            # Resize heatmap
            heatmap = cv2.resize(heatmap, (224,224))
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
            heatmap = heatmap.astype(np.float32) / 255

            overlay = heatmap * 0.4 + img

            # Plot Original
            plt.subplot(num_images, 2, 2*count + 1)
            plt.imshow(img)
            plt.title("Original Stroke")
            plt.axis("off")

            # Plot Grad-CAM
            plt.subplot(num_images, 2, 2*count + 2)
            plt.imshow(overlay)
            plt.title("Grad-CAM")
            plt.axis("off")

            count += 1

            if count == num_images:
                break
    if count == num_images:
        break

plt.tight_layout()
plt.show()

5-FOLD CROSS VALIDATION IMPLEMENTATION


In [ ]:
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
dataset = datasets.ImageFolder(data_dir, transform=transform)
labels = np.array([dataset.samples[i][1] for i in range(len(dataset))])

In [ ]:
fold_results = {
    "accuracy": [],
    "recall": [],
    "f1": [],
    "auc": []
}

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(len(dataset)), labels)):

    print(f"\n===== FOLD {fold+1} =====")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    # Initialize fresh model each fold
    model = EfficientNet_CBAM().to(device)

    # Weighted loss
    num_normal = np.sum(labels[train_idx] == 0)
    num_stroke = np.sum(labels[train_idx] == 1)
    pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=5e-5)

    epochs = 10  # keep smaller for CV (you can increase)

    for epoch in range(epochs):

        model.train()
        for images, labels_batch in train_loader:

            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            optimizer.zero_grad()

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    # ---- VALIDATION ----
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:

            images = images.to(device)
            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy().ravel())
            all_labels.extend(labels_batch.numpy())
            all_probs.extend(probs.cpu().numpy().ravel())

    acc = accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    fold_results["accuracy"].append(acc)
    fold_results["recall"].append(rec)
    fold_results["f1"].append(f1)
    fold_results["auc"].append(auc)

    print(f"Fold {fold+1} Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"AUC: {auc:.4f}")

In [ ]:
print("\n===== FINAL 5-FOLD RESULTS =====")

for metric in fold_results:
    values = fold_results[metric]
    print(f"{metric.upper()}: {np.mean(values):.4f} ± {np.std(values):.4f}")